# Explaining and Testing the Regime Reversal

We found that low volatility periods have better coverage at day 0, but
high volatility periods have better coverage at day 29. Two things to check:

1. Does interval width explain this? (maybe high-vol forecasts are just wider)
2. Is this difference statistically significant, or could it be noise?

In [1]:
import numpy as np

In [2]:
sensex_forecasts = np.load("../data/forecasts.npy")
sensex_actuals = np.load("../data/actuals.npy")
sensex_regimes = np.load("../data/sensex_regime_labels.npy", allow_pickle=True)

nifty_forecasts = np.load("../data/nifty_forecasts.npy")
nifty_actuals = np.load("../data/nifty_actuals.npy")
nifty_regimes = np.load("../data/nifty_regime_labels.npy", allow_pickle=True)

## Part 1: Compare average interval width by regime

In [3]:
def width_by_regime(forecasts, regimes, day):
    day_forecasts = forecasts[:, day, :]
    lower = day_forecasts[:, 1]
    upper = day_forecasts[:, 9]
    width = upper - lower

    results = {}
    for regime in ["low_vol", "high_vol"]:
        mask = (regimes == regime)
        results[regime] = width[mask].mean()
    return results

In [4]:
print("SENSEX interval width")
for day in [0, 29]:
    w = width_by_regime(sensex_forecasts, sensex_regimes, day)
    print(f"day {day}: low_vol width = {round(w['low_vol'],1)}   high_vol width = {round(w['high_vol'],1)}")

print()
print("NIFTY interval width")
for day in [0, 29]:
    w = width_by_regime(nifty_forecasts, nifty_regimes, day)
    print(f"day {day}: low_vol width = {round(w['low_vol'],1)}   high_vol width = {round(w['high_vol'],1)}")

SENSEX interval width
day 0: low_vol width = 972.5   high_vol width = 1031.0999755859375
day 29: low_vol width = 4923.60009765625   high_vol width = 5673.39990234375

NIFTY interval width
day 0: low_vol width = 282.3999938964844   high_vol width = 275.79998779296875
day 29: low_vol width = 1446.0   high_vol width = 1502.699951171875


If high_vol width is a lot bigger than low_vol width, especially at day 29,
that supports our explanation: the model already outputs wider intervals in
high volatility periods, and that extra width happens to help coverage more
at longer horizons.

## Part 2: Is the coverage difference statistically significant?

We use bootstrap resampling: repeatedly resample each regime's windows with
replacement, recompute coverage each time, and build a distribution of the
difference (low_vol coverage minus high_vol coverage).

If the 95% range of that distribution does not include zero, the difference
is unlikely to be just random noise.

In [5]:
def bootstrap_coverage_diff(forecasts, actuals, regimes, day, n_boot=2000):
    day_forecasts = forecasts[:, day, :]
    day_actuals = actuals[:, day]

    lower = day_forecasts[:, 1]
    upper = day_forecasts[:, 9]
    inside = (day_actuals >= lower) & (day_actuals <= upper)

    low_inside = inside[regimes == "low_vol"]
    high_inside = inside[regimes == "high_vol"]

    diffs = []
    for _ in range(n_boot):
        low_sample = np.random.choice(low_inside, size=len(low_inside), replace=True)
        high_sample = np.random.choice(high_inside, size=len(high_inside), replace=True)
        diff = low_sample.mean() - high_sample.mean()
        diffs.append(diff)

    diffs = np.array(diffs)
    ci_low = np.percentile(diffs, 2.5)
    ci_high = np.percentile(diffs, 97.5)

    return diffs.mean(), ci_low, ci_high

In [6]:
print("SENSEX: low_vol coverage - high_vol coverage")
for day in [0, 29]:
    mean_diff, ci_low, ci_high = bootstrap_coverage_diff(sensex_forecasts, sensex_actuals, sensex_regimes, day)
    print(f"day {day}: mean diff = {round(mean_diff*100,1)} pts   95% CI = [{round(ci_low*100,1)}, {round(ci_high*100,1)}]")

SENSEX: low_vol coverage - high_vol coverage
day 0: mean diff = 6.9 pts   95% CI = [-1.1, 15.1]
day 29: mean diff = -9.9 pts   95% CI = [-18.9, -0.8]


In [7]:
print("NIFTY: low_vol coverage - high_vol coverage")
for day in [0, 29]:
    mean_diff, ci_low, ci_high = bootstrap_coverage_diff(nifty_forecasts, nifty_actuals, nifty_regimes, day)
    print(f"day {day}: mean diff = {round(mean_diff*100,1)} pts   95% CI = [{round(ci_low*100,1)}, {round(ci_high*100,1)}]")

NIFTY: low_vol coverage - high_vol coverage
day 0: mean diff = 11.2 pts   95% CI = [3.6, 18.9]
day 29: mean diff = -8.2 pts   95% CI = [-16.9, 0.1]


## How to read the confidence intervals

If the 95% CI does not cross zero, the difference is statistically significant
at the 5% level, meaning it's unlikely to just be random sampling noise.

If the CI does cross zero, we cannot rule out that the difference happened
by chance, and we should describe it as a suggestive but not confirmed pattern.